In [2]:
# Exporting the SQL views to CSV for Tableau

In [3]:
# Bring in the tools and connect to the project database
# (password typed into a hidden box, never saved)

import pandas as pd
from sqlalchemy import create_engine, URL
import getpass
import os

password = getpass.getpass("MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="ev_charging_gap"
)

engine = create_engine(connection_url)

MySQL password:  ········


In [4]:
# Export the three SQL views to CSV for Tableau
# 1. Create a data/tableau folder (exist_ok=True: no error if it's already there)
# 2. Read each view from MySQL, exactly like reading a table
# 3. Save each one as a CSV, without row numbers
# 4. Check the shapes: expect (10108, 9), (361, 15) and (18, 7)

os.makedirs("../data/tableau", exist_ok=True)

timeseries = pd.read_sql("SELECT * FROM v_council_timeseries", engine)
timeseries.to_csv("../data/tableau/council_timeseries.csv", index=False)

summary = pd.read_sql("SELECT * FROM v_council_summary", engine)
summary.to_csv("../data/tableau/council_summary.csv", index=False)

europe = pd.read_sql("SELECT * FROM v_europe_2025", engine)
europe.to_csv("../data/tableau/europe_2025.csv", index=False)

print(timeseries.shape)
print(summary.shape)
print(europe.shape)

(10108, 9)
(361, 15)
(18, 7)


In [5]:
# Fix two council codes so every council matches the 2025 map boundaries
# 1. Barnsley and Sheffield have new ONS codes (likely a small boundary change);
#    DfT still uses the old ones
# 2. Add a map_code column: the new code for those two, the same code for everyone else
#    (area_code stays unchanged, so the SQL and data stay traceable)
# 3. Re-save both council files for Tableau
# 4. Check: expect 361 matches with the map

code_fix = {
    "E08000016": "E08000038",   # Barnsley
    "E08000019": "E08000039"    # Sheffield
}

summary = pd.read_csv("../data/tableau/council_summary.csv")
timeseries = pd.read_csv("../data/tableau/council_timeseries.csv")

summary["map_code"] = summary["area_code"].replace(code_fix)
timeseries["map_code"] = timeseries["area_code"].replace(code_fix)

summary.to_csv("../data/tableau/council_summary.csv", index=False)
timeseries.to_csv("../data/tableau/council_timeseries.csv", index=False)

boundaries = pd.read_csv("../data/boundaries/lad_may_2025_uk_buc.csv")
print(len(set(boundaries["LAD25CD"]) & set(summary["map_code"])))

361
